# micrograd exercises

1. watch the [micrograd video](https://www.youtube.com/watch?v=VMj-3S1tku0) on YouTube
2. come back and complete these exercises to level up :)

## section 1: derivatives

In [1]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos

def f(a, b, c):
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))

6.336362190988558


In [2]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29

def gradf(a, b, c):
  fa = -3*a**2 -0.5*a**-0.5
  fb = cos(3*b)*3 + 2.5*b**1.5
  fc = -1*-1*c**-2
  return [fa, fb, fc] # todo, return [df/da, df/db, df/dc]

# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390593273
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027111255
OK for dim 2: expected 0.0625, yours returns 0.0625


In [3]:
# now estimate the gradient numerically without any calculus, using
# the approximation we used in the video.
# you should not call the function df from the last cell

# -----------
h = 0.0000005
a = 2
b = 3
c = 4
nfa = (f(a+h,b,c) - f(a,b,c)) / h
nfb = (f(a,b+h,c) - f(a,b,c)) / h
nfc = (f(a,b,c+h) - f(a,b,c)) / h
numerical_grad = [nfa, nfb, nfc] # TODO
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353556368083218
OK for dim 1: expected 10.25699027111255, yours returns 10.256990970347601
OK for dim 2: expected 0.0625, yours returns 0.06249999273677531


In [4]:
# there is an alternative formula that provides a much better numerical
# approximation to the derivative of a function.
# learn about it here: https://en.wikipedia.org/wiki/Symmetric_derivative
# implement it. confirm that for the same step size h this version gives a
# better approximation.

# -----------
h = 0.0000005
a = 2
b = 3
c = 4
nfa2 = (f(a+h,b,c) - f(a-h,b,c)) / (2*h)
nfb2 = (f(a,b+h,c) - f(a,b-h,c)) / (2*h)
nfc2 = (f(a,b,c+h) - f(a,b,c-h)) / (2*h)
numerical_grad2 = [nfa2, nfb2, nfc2] # TODO
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")

for i, (x,y) in enumerate(zip(numerical_grad2, gradf(a,b,c))):
  print(i, ':', abs(x-y))

for i, (x,y) in enumerate(zip(numerical_grad, gradf(a,b,c))):
  print(i, ':', abs(x-y))


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390909155
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027401572
OK for dim 2: expected 0.0625, yours returns 0.06250000073038109
0 : 3.1588243132318894e-10
1 : 2.9031710369054053e-09
2 : 7.303810889425222e-10
0 : 2.977489945266143e-06
1 : 6.992350520818036e-07
2 : 7.263224688358605e-09


## section 2: support for softmax

In [5]:
# Value class starter code, with many functions taken out
from math import exp, log

class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other): # exactly as in the video
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  def __radd__(self, other):
    return self + other

  # a/b
  def __truediv__(self, other):
    out = Value(self.data / other.data, (self, other), '/')

    # L(d)
    # d(a,b) = a/b = a * b^-1
    # dL/da = dL/dd * dd/da
    # dL/db = dL/dd * dd/db

    def _backward():
      self.grad += out.grad * (1/other.data)
      # other.grad += out.grad * (-1 * self.data * (1/(other.data**2)))
      other.grad += out.grad * (self.data * (-1) * (other.data ** (-2)))
    out._backward = _backward

    return out

  # x -> -x
  def __neg__(self):
    out = Value(-1*self.data, (self,), '-1')

    def _backward():
      self.grad += out.grad * (-1)
    out._backward = _backward

    return out

  # x -> exp(x)
  def exp(self):
    out = Value(exp(self.data), (self,), 'exp')

    # L(e)
    # e(x)
    # dL/dx = dL/de * de/dx
    # ^self   ^out   ^local
    def _backward():
      self.grad += out.grad * exp(self.data)
    out._backward = _backward

    return out

  # x -> log(x)
  def log(self):
    out = Value(log(self.data), (self,), 'log')

    def _backward():
      self.grad += out.grad * (1/self.data)
    out._backward = _backward

    return out

  # ------
  # re-implement all the other functions needed for the exercises below
  # your code here
  # TODO
  # exp
  # a/b
  # log
  # ------

  def backward(self): # exactly as in video
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [6]:
# without referencing our code/video __too__ much, make this cell work
# you'll have to implement (in some cases re-implemented) a number of functions
# of the Value object, similar to what we've seen in the video.
# instead of the squared error loss this implements the negative log likelihood
# loss, which is very often used in classification.

# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
  counts = [logit.exp() for logit in logits]
  denominator = sum(counts)
  out = [c / denominator for c in counts]
  return out

# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log() # dim 3 acts as the label for this input example
loss.backward()

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")


OK for dim 0: expected 0.041772570515350445, yours returns 0.041772570515350445
OK for dim 1: expected 0.8390245074625319, yours returns 0.8390245074625319
OK for dim 2: expected 0.005653302662216329, yours returns 0.005653302662216329
OK for dim 3: expected -0.8864503806400986, yours returns -0.886450380640099


In [7]:
# verify the gradient using the torch library
# torch should give you the exact same gradient
import torch

logits = torch.tensor(
    [0.0, 3.0, -2.0, 1.0],
    requires_grad=True
)
probs = torch.softmax(logits, dim=0)
l = -probs[3].log()
l.backward()
logits.grad

tensor([ 0.0418,  0.8390,  0.0057, -0.8865])

In [8]:
x = torch.tensor(3.0)
print(x)
x = torch.tensor([1.0, 2.0, 3.0])
print(x)

tensor(3.)
tensor([1., 2., 3.])


In [9]:
x = torch.tensor(2.0, requires_grad=True)

a = x.exp()
b = x.log()
L = a + b

L.backward()
print(x.grad)

dLda = 1
dLdb = 1
dadx = exp(x.item())
dbdx = 1/(x.item())

dLdx = dLda * dadx + dLdb * dbdx
print(dLdx)

tensor(7.8891)
7.88905609893065


In [10]:
logits = torch.tensor(
    [0.0, 3.0, -2.0, 1.0],
    requires_grad=True
)
probs = torch.softmax(logits, dim=0)
l = -probs[3].log()

print(logits)
print(probs)
print(probs.sum())
print(l)

l.backward()

print(logits.grad)


tensor([ 0.,  3., -2.,  1.], requires_grad=True)
tensor([0.0418, 0.8390, 0.0057, 0.1135], grad_fn=<SoftmaxBackward0>)
tensor(1.0000, grad_fn=<SumBackward0>)
tensor(2.1755, grad_fn=<NegBackward0>)
tensor([ 0.0418,  0.8390,  0.0057, -0.8865])
